# 📨 Day 04a: Message Queues — Kafka, RabbitMQ & Event-Driven

---

## 🎯 What You'll Master Today
- Why message queues (decoupling, async processing)
- Kafka vs RabbitMQ vs SQS
- Pub/Sub pattern
- Event-driven architecture
- Exactly-once vs at-least-once delivery

**The analogy:** A message queue is like a POST OFFICE — senders drop off letters, recipients pick them up at their own pace. Sender doesn't wait for recipient.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  WHY MESSAGE QUEUES?                                             ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Without queue (synchronous):                                    ║
║  User → API → Process Payment → Send Email → Update Analytics   ║
║                                                                   ║
║  Total time: 200 + 500 + 300 = 1000ms 😱                        ║
║  If email fails → everything fails!                              ║
║                                                                   ║
║  With queue (asynchronous):                                      ║
║  User → API → Process Payment → Return 200                      ║
║                   │                                               ║
║                   ├──► Queue ──► Email Service                   ║
║                   └──► Queue ──► Analytics Service               ║
║                                                                   ║
║  Total time: 200ms ⚡ (non-critical work is async)               ║
║  Email fails? Queue retries later!                               ║
║                                                                   ║
║  Benefits:                                                       ║
║  ✅ Decoupling (services don't know about each other)            ║
║  ✅ Async processing (fast response + background work)           ║
║  ✅ Load leveling (absorb traffic spikes)                        ║
║  ✅ Reliability (messages persist, retry on failure)             ║
║  ✅ Scale consumers independently                                ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Queue vs Pub/Sub vs Streaming
# ============================================================

print("""
THREE MESSAGING PATTERNS
════════════════════════

1️⃣ POINT-TO-POINT QUEUE
   Producer → [Queue] → Consumer
   
   • One message → one consumer
   • Message removed after processing
   • Example: Task queue (worker processes jobs)
   • Tools: RabbitMQ, SQS, Celery

2️⃣ PUB/SUB (Publish-Subscribe)
   Publisher → [Topic] ──► Subscriber A
                       ──► Subscriber B  
                       ──► Subscriber C
   
   • One message → ALL subscribers
   • Message is broadcast (fan-out)
   • Example: Notifications (email + push + SMS)
   • Tools: Redis Pub/Sub, SNS, Kafka topics

3️⃣ EVENT STREAMING
   Producer → [Log] ← Consumer Group A (reads at own pace)
                     ← Consumer Group B (reads at own pace)
   
   • Messages are RETAINED (not deleted after consumption)
   • Consumers track their own offset (position in log)
   • Can replay events from any point in time
   • Example: Event sourcing, analytics, CDC
   • Tools: Kafka, AWS Kinesis, Pulsar
""")

In [ ]:
# ============================================================
# 2. Kafka — The Event Streaming King
# ============================================================

print("""
APACHE KAFKA — ARCHITECTURE
═══════════════════════════

Key concepts:
  Topic    = named stream of events ("orders", "user-events")
  Partition = ordered, immutable log within a topic
  Offset   = position of a message in a partition
  Broker   = Kafka server (cluster of N brokers)
  Producer = writes events to topics
  Consumer = reads events from topics
  Consumer Group = set of consumers that share partitions

  Topic "orders" with 3 partitions:
  ┌───────────────────────────────────┐
  │ P0: [msg0][msg1][msg2][msg3]→     │
  │ P1: [msg0][msg1][msg2]→           │
  │ P2: [msg0][msg1][msg2][msg3][msg4]│
  └───────────────────────────────────┘
  
  Consumer Group A:          Consumer Group B:
  C1 reads P0, P1            C3 reads P0
  C2 reads P2                C4 reads P1, P2

  Each partition → only ONE consumer in a group
  More partitions = more parallelism
  
Why Kafka is fast:
  • Sequential I/O (append-only log) — faster than random I/O
  • Zero-copy (sendfile syscall) — no user-space copying
  • Batching — producers batch messages before sending
  • Page cache — OS-level caching
  • Result: millions of messages/sec on commodity hardware

Retention:
  • Time-based: keep messages for 7 days (default)
  • Size-based: keep 1 TB per partition
  • Compacted: keep latest value per key (changelog)
""")

In [ ]:
# ============================================================
# 3. Kafka vs RabbitMQ vs SQS
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════════╗
║  KAFKA vs RABBITMQ vs SQS                                        ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  Feature      │ Kafka           │ RabbitMQ        │ AWS SQS       ║
║  ─────────────│─────────────────│─────────────────│───────────    ║
║  Model        │ Event streaming │ Message broker  │ Queue (managed║
║  Message life │ Retained (days) │ Deleted on ACK  │ Deleted on ACK║
║  Ordering     │ Per partition   │ Per queue       │ Best-effort * ║
║  Throughput   │ Millions/sec    │ Tens of K/sec   │ Thousands/sec ║
║  Replay       │ ✅ Yes (offset) │ ❌ No           │ ❌ No         ║
║  Routing      │ Topic+partition │ Exchange+routing│ Simple queue  ║
║  Delivery     │ At-least-once   │ At-most/least   │ At-least-once ║
║  Complexity   │ High            │ Medium          │ Low (managed) ║
║  Use case     │ Event sourcing, │ Task queues,    │ Simple async, ║
║               │ CDC, analytics  │ RPC, routing    │ AWS workflows ║
║                                                                    ║
║  * SQS FIFO queues provide strict ordering                        ║
║                                                                    ║
║  Decision guide:                                                   ║
║  • Need replay / event sourcing → Kafka                           ║
║  • Need complex routing / RPC → RabbitMQ                          ║
║  • Need managed + simple → SQS                                    ║
║  • Need millions of events/sec → Kafka                            ║
║  • Need exactly-once with idempotency → Kafka (transactions)     ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ============================================================
# 4. Delivery Guarantees
# ============================================================

print("""
DELIVERY GUARANTEES — CRITICAL INTERVIEW TOPIC
═══════════════════════════════════════════════

Three levels:

1️⃣ AT-MOST-ONCE (fire and forget)
   Producer sends → doesn't wait for ACK
   Message may be lost, never duplicated
   Use: Logging, metrics (losing one metric is OK)

2️⃣ AT-LEAST-ONCE (default in most systems)
   Producer sends → waits for ACK → retries if no ACK
   Message never lost, but may be duplicated
   Use: Payments (but need idempotency!)

3️⃣ EXACTLY-ONCE (hardest to achieve)
   Each message processed exactly once
   How: Kafka transactions + idempotent producer + consumer offset commit
   Or: At-least-once + idempotent consumer (same result!)
   Use: Financial transactions, inventory updates

💡 Interview insight:
   "True exactly-once is very hard. In practice, I'd use
    at-least-once delivery + idempotent processing."

Idempotency example:
  ❌ charge_user(amount)       → calling twice = charged twice
  ✅ charge_user(txn_id, amt)  → calling twice = charged once
     (DB has unique constraint on txn_id)
""")

In [ ]:
# ============================================================
# 5. Event-Driven Architecture Patterns
# ============================================================

print("""
EVENT-DRIVEN ARCHITECTURE
═════════════════════════

Pattern 1: EVENT NOTIFICATION
  OrderService publishes: {type: "order.created", orderId: 123}
  → EmailService sends confirmation
  → InventoryService reduces stock
  → AnalyticsService records sale
  Each service reacts independently!

Pattern 2: EVENT SOURCING
  Don't store current state — store ALL events.
  Account balance = replay all deposits/withdrawals
  
  Events: [+100] [+50] [-30] [+200] [-75] = Balance: $245
  
  Benefits: Full audit trail, can rebuild state at any point
  Used by: Banking, trading systems, Git!

Pattern 3: CQRS (Command Query Responsibility Segregation)
  Separate write model from read model.
  
  Commands (writes) → Write DB (normalized)
         │
         └→ Events → Read DB (denormalized, optimized for queries)
  
  Example: E-commerce
    Write: normalized order tables
    Read: denormalized order summary (pre-joined for dashboard)

Pattern 4: SAGA (distributed transactions)
  Order → Payment → Inventory → Shipping
  If any step fails → compensating transactions (undo)
  
  Choreography: each service listens + publishes events
  Orchestration: central coordinator tells each service what to do
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Kafka vs RabbitMQ? | Kafka = streaming, retains events, replay. RabbitMQ = broker, complex routing, task queues |
| 2 | At-least-once delivery? | Retry until ACK. May duplicate. Fix: idempotent consumers (unique txn_id) |
| 3 | What is event sourcing? | Store events not state. Rebuild state by replaying. Full audit trail |
| 4 | CQRS — when to use? | Read/write patterns differ drastically. Separate optimized read models |
| 5 | Why Kafka is fast? | Sequential I/O, zero-copy, batching, page cache, append-only log |
| 6 | Saga pattern? | Distributed transactions via events. Choreography (event) vs Orchestration (central) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Queues decouple services + enable async processing    │
## │  • Kafka = streaming (replay), RabbitMQ = broker (route) │
## │  • At-least-once + idempotency ≈ exactly-once            │
## │  • Event sourcing: store events, derive state            │
## │  • CQRS: separate read and write models                  │
## │  • Saga for distributed transactions                     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **API Gateway, Rate Limiting & Circuit Breaker**